# L6c: Stationary Iterative Methods for Linear Systems

> **Learning objectives**
>
> - Split a linear system into diagonal, lower, and upper components.
> - Distinguish Jacobi, Gauss–Seidel, and SOR updates.
> - Use residual norms and iteration matrices to reason about convergence.
> - Recognize that diagonal dominance is sufficient, not necessary.


## Setup


In [1]:
include(joinpath(@__DIR__, "Include.jl"))


## General
Suppose we have a _square system_ of linear equations represented in matrix form as $\mathbf{A}\mathbf{x} = \mathbf{b}$,
where the system matrix $\mathbf{A}\in\mathbb{R}^{n \times n}$, the unknown vector is $\mathbf{x}\in\mathbb{R}^{n}$, and $\mathbf{b}\in\mathbb{R}^{n}$ is the right-hand side vector. Our goal is to _iteratively_ find a vector $\mathbf{x}$ that _approximately satisfies_ this equation.

Let's sketch the steps of a general iterative method for solving this linear system.

__Initialize__: Given the system matrix $\mathbf{A}\in\mathbb{R}^{n \times n}$ and the right-hand side vector $\mathbf{b}\in\mathbb{R}^{n}$, we start with an initial guess for the solution vector $\mathbf{x}^{(0)}\in\mathbb{R}^{n}$. This guess can be a zero vector or any other reasonable approximation. Set $\texttt{converged} \gets \texttt{false}$ and the iteration counter $k\gets0$. Specify a convergence criterion, such as a tolerance level $\epsilon > 0$, and a maximum number of iterations $\texttt{maxiter}$.

While not $\texttt{converged}$ __do__:
1. Calculate the residual vector $\mathbf{r}^{(k)} \gets \mathbf{b} - \mathbf{A}\mathbf{x}^{(k)}$.
2. Check for convergence:
   - If $\|\mathbf{r}^{(k)}\|_{2} < \epsilon$, __then__: set $\texttt{converged} \gets \texttt{true}$. Return the current estimate $\mathbf{x}^{(k)}$ as the approximate solution.
   - If $k \geq \texttt{maxiter}$, __then__: set $\texttt{converged} \gets \texttt{true}$, and print a __warning__ that the method did not converge. Return the current estimate $\mathbf{x}^{(k)}$ as the approximate solution.
3. Calculate an update direction. Split $\mathbf{A} = \mathbf{M} - \mathbf{N}$, where $\mathbf{M}$ is a matrix that can be easily inverted (e.g., diagonal or triangular), and $\mathbf{N}$ is the remaining part of the matrix. The update direction can then be computed as: $\mathbf{d}^{(k)} \gets \mathbf{M}^{-1}\mathbf{r}^{(k)}$.
4. Update the solution vector: $\mathbf{x}^{(k+1)} \gets \mathbf{x}^{(k)} + \mathbf{d}^{(k)}$.
5. Increment the iteration counter: $k \gets k + 1$.

___


## Update direction
The magic of iterative methods lies in the choice of the update direction. But where does this update direction come from? In the general case, we can think of the update direction as being derived from the residual vector $\mathbf{r}^{(k)}$ and the system matrix $\mathbf{A}$. 

Starting from the original system:
$$
\begin{align*}
\mathbf{A}\;\mathbf{x} &= \mathbf{b}\quad\Longrightarrow\text{substitute}\;\mathbf{A} = \mathbf{M} - \mathbf{N} \\
(\mathbf{M} - \mathbf{N})\;\mathbf{x} &= \mathbf{b} \\
\mathbf{M}\mathbf{x}^{(k+1)} - \mathbf{N}\mathbf{x}^{(k)} &= \mathbf{b}\quad\Longrightarrow\text{solve for}\;\mathbf{x}^{(k+1)}\\
\mathbf{x}^{(k+1)} &= \mathbf{M}^{-1}(\mathbf{b} + \mathbf{N}\mathbf{x}^{(k)})\quad\Longrightarrow\text{substitute}\;\mathbf{M}^{-1}\mathbf{N} = \mathbf{I} - \mathbf{M}^{-1}\mathbf{A}\\
\mathbf{x}^{(k+1)} &= \mathbf{x}^{(k)} + \mathbf{M}^{-1}\underbrace{(\mathbf{b} - \mathbf{A}\mathbf{x}^{(k)})}_{\text{residual}\;\mathbf{r}^{(k)}}\\
\mathbf{x}^{(k+1)} & = \mathbf{x}^{(k)} + \underbrace{\mathbf{M}^{-1}\mathbf{r}^{(k)}}_{\text{direction}\;\mathbf{d}^{(k)}}\\
\mathbf{x}^{(k+1)} &= \mathbf{x}^{(k)} + \mathbf{d}^{(k)}\quad\blacksquare\\
\end{align*}
$$

The update direction $\mathbf{d}^{(k)}$ is derived from the residual vector $\mathbf{r}^{(k)}$ and the system matrix $\mathbf{A}$. The choice of $\mathbf{M}$ and $\mathbf{N}$ determines the specific iterative method being used, such as Jacobi, Gauss-Seidel, or Successive Over-Relaxation (SOR).

___


## Convergence
One common question that arises when using iterative methods is: _When does the method converge?_ In other words, how do we know that the sequence of iterates $\{\mathbf{x}^{(k)}\}$ will approach the true solution $\mathbf{x}^{\star}$ as $k$ increases? A stationary iteration $\mathbf{x}^{(k+1)}=\mathbf{G}\,\mathbf{x}^{(k)}+\mathbf{c}$ converges __for every initial guess__ $\mathbf{x}^{(0)}$ __if and only if__:
$$
\rho(\mathbf{G}) = \rho\bigl(\mathbf{M}^{-1}\mathbf{N}\bigr) \;<\;1.
$$
The spectral radius $\rho(\mathbf{G}) = \;\max_i|\lambda_i|$, where $\lambda_i$ are the eigenvalues of the iteration matrix $\mathbf{G}$. This means that the method will converge to the true solution regardless of the initial guess $\mathbf{x}^{(0)}$.

> __Additional Notes:__ The derivation of the spectral radius convergence condition is provided in the [Advanced: Where does spectral radius convergence condition come from?](CHEME-5800-L6c-Advanced-Convergence-IterativeMethods-Fall-2026.ipynb) notebook. Check it out if you are interested in the mathematical details behind this condition.

### Diagonal dominance

If the coefficient matrix $\mathbf{A}$ is **strictly diagonally dominant**, Jacobi and Gauss–Seidel iterations converge from any initial guess, so diagonal dominance is **sufficient but not necessary** condition for convergence.

Recall that we split the matrix $\mathbf{A}$ into its diagonal and off-diagonal components: $\mathbf{A} = \mathbf{M} - \mathbf{N}$, where $\mathbf{M}$ is the diagonal part of $\mathbf{A}$ and $\mathbf{N}$ is the off-diagonal part. The iteration matrix is then given by: $\mathbf{G} = \mathbf{M}^{-1}\mathbf{N}$. Thus, for a matrix $\mathbf{A}$ to be strictly diagonally dominant, we require that for each row $i$:
$$
    |\mathbf{M}_{ii}| \;>\; \sum_{j\neq i} |\mathbf{N}_{ij}|,
$$
This is related to the spectral radius of the iteration matrix $\mathbf{G}$ through the induced infinity-norm:
$$
    \|\mathbf{G}\|_\infty 
    = \max \left\{\sum_j \bigl| (\mathbf{M}^{-1}\mathbf{N})_{ij}\bigr|\right\}_{i} 
    \;\le\;\max_i \frac{1}{|\mathbf{M}_{ii}|}\sum_{j\neq i}|\mathbf{N}_{ij}|
    \;<\;1.
$$
A basic property of induced norms is that for any eigenpair $(\lambda, \mathbf{v})$ of $\mathbf{G}$ with eigenvector $\mathbf{v}\neq0$:
$$
|\lambda|\;\|\mathbf{v}\|_\infty \;=\;\|\lambda \mathbf{v}\|_\infty
\;=\;\|\mathbf{G}\mathbf{v}\|_\infty
\;\le\;\|\mathbf{G}\|_\infty\,\|\mathbf{v}\|_\infty
\quad\Longrightarrow\quad
|\lambda|\le\|\mathbf{G}\|_\infty.
$$
Taking the maximum over all eigenvalues gives
$\rho(\mathbf{G})=\max_i|\lambda_i|\le\|\mathbf{G}\|_\infty$.


### Rate of convergence?
The rate of convergence of an iterative method is related to the spectral radius of the iteration matrix $\mathbf{G}$. Let's first define the error vector at iteration $k$ as:
$$
\mathbf{e}^{(k)} = \mathbf{x}^{(k)} - \mathbf{x}^{\star}
$$
where $\mathbf{x}^{(k)}$ is the current iterate and $\mathbf{x}^{\star}$ is the true solution. Now, let's consider a _worst-case_ bound. Suppose we have some (induced) norm $\|\mathbf{G}\| = q$; then for any iteration $k$ we have:
$$\begin{align*}
\|\mathbf{e}^{(k)}\| & = \lVert\mathbf{G}^{k}\;\mathbf{e}^{(0)}\rVert\\
& \le\; \|\mathbf{G}^{k}\|\,\|\mathbf{e}^{(0)}\|\\
& \le\; q^{k}\,\|\mathbf{e}^{(0)}\|\\
\end{align*}
$$
To reach a desired accuracy $\epsilon$ at iteration $k$, we can bound the error as:
$$
\|\mathbf{e}^{(k)}\| \leq q^{k}\,\|\mathbf{e}^{(0)}\| \leq \epsilon
$$
which implies:
$$
\boxed{
   k\geq\frac{\ln(\epsilon/\|\mathbf{e}^{(0)}\|)}{|\ln(\lVert\mathbf{G}\rVert)|}
\quad\blacksquare}
$$

___


## Specific Methods
There are several specific iterative methods that can be used to solve systems of linear equations.

* __Jacobi Method__: The Jacobi method is an iterative solver that, starting from an initial guess, repeatedly refines each unknown in parallel using the residual between the right-hand side and the contributions from other variables. It is easy to implement and parallelize, and converges when the system matrix satisfies appropriate conditions (e.g., strict diagonal dominance). [Let's check out the algorithm.](CHEME-5800-L6c-Algorithm-JacobiMethod-Fall-2026.ipynb)

* __Gauss–Seidel Method__: The Gauss–Seidel method is an iterative solver that, starting from an initial guess, refines each unknown one at a time using the most recent updates, immediately incorporating new values as they become available. This approach typically yields faster convergence than the Jacobi method under the same matrix conditions. It is simple to implement but inherently sequential, and converges when the system matrix is, for example, strictly diagonally dominant. [Let's check out the algorithm.](CHEME-5800-L6c-Algorithm-GaussSeidel-Fall-2026.ipynb)

* __Successive Over-Relaxation (SOR) Method__: The SOR method builds on Gauss–Seidel by introducing a relaxation factor $\omega\in(0,2)$ that _over-relaxes_ each update by blending the new Gauss–Seidel value with the previous iterate to further accelerate convergence. With a well-chosen $\omega$, SOR can dramatically improve the solution speed for diagonally-dominant systems, with convergence guaranteed for $0<\omega<2$. [Let's check out the algorithm.](CHEME-5800-L6c-Algorithm-SOR-Fall-2026.ipynb)

Let's look at an example of each of these methods in action!

> __Example__
>
> [▶ Fun with Iterative Methods](CHEME-5800-L6c-Example-FunWithIterativeSolvers-Fall-2026.ipynb). In this example, we will explore the implementation of various iterative methods for solving (square) systems of linear equations. We will compare the performance of these methods on randomly generated matrices and analyze their convergence behavior.

___


## Lab
In L6b, we will take a deeper dive into the implementation of these iterative methods and compare their performance on a classical test problem: a system of linear equations arising from the discretization of the 2D Poisson partial differential equation.


## Summary
In this lecture, we explored iterative methods for solving linear systems, focusing on the general algorithm, convergence conditions, and specific implementations.

> __Key Takeaways:__
>
> - **General Algorithm**: Iterative methods start with an initial guess and update the solution using residuals, checking for convergence based on tolerance or iteration limits.
> - **Convergence Criteria**: Methods converge when the spectral radius of the iteration matrix is less than 1, which is related to diagonal dominance of the system matrix.
> - **Specific Methods**: Jacobi, Gauss-Seidel, and SOR differ in how they compute updates, with SOR using a relaxation factor for potentially faster convergence.

___


## Convergence checkpoint

For a stationary iteration $x^{(k+1)}=T x^{(k)}+c$, convergence for every initial guess occurs when the spectral radius $\rho(T)<1$. Diagonal dominance is an easy structural condition that often implies this contract; the residual $\|Ax^{(k)}-b\|$ remains the computational check.


In [2]:
A = [4.0 -1.0 0.0; -1.0 4.0 -1.0; 0.0 -1.0 3.0]
D = Diagonal(diag(A))
T_jacobi = Matrix(I, 3, 3) - D \ A
spectral_radius(T_jacobi)


0.3818813079129867

## Summary

Jacobi uses only the previous iterate, Gauss–Seidel immediately reuses updated components, and SOR relaxes that update. None should be accepted without a convergence flag and residual check.
